# Lab 24 — MLP Deep Learning với Nested LOCO

Mục tiêu: thử một mạng neural network nhỏ trên cùng 920 dòng UCI, giữ nguyên P1 sentinel-aware và thiết kế nested LOCO của Lab 22.

- `F0_P1_fixed`: kiến trúc MLP cố định.
- `F0_P1_optuna_nested`: Optuna tối ưu kiến trúc và regularization bên trong các bệnh viện huấn luyện.
- Early stopping được dùng trong từng inner fold để hạn chế overfitting.
- Threshold giữ ở `0.50`; calibration và threshold tuning sẽ làm ở lab riêng.
- MLP chỉ là mô hình đối chứng; không thay thế Logistic Regression/LightGBM nếu LOCO không chứng minh được lợi ích.

In [ ]:
!pip -q install optuna lightgbm seaborn

import json
import random
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, average_precision_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
RANDOM_STATE = 42
N_TRIALS = 30
MAX_EPOCHS = 150
PATIENCE = 15
THRESHOLD = 0.50
EVAL_SEEDS = (42, 123, 2025)
OUTPUT_DIR = Path('/content/uci_multicenter_mlp_loco_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
            'exang','oldpeak','slope','ca','thal']
TARGET = 'target'
NUMERICAL_FEATURES = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL_FEATURES = ['sex','cp','fbs','restecg','exang','slope','ca','thal']
BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {'cleveland':'processed.cleveland.data', 'hungarian':'processed.hungarian.data',
         'switzerland':'processed.switzerland.data', 'va':'processed.va.data'}
COLUMNS = FEATURES + ['num']
LOCAL_DATA_DIR_CANDIDATES = [
    Path('/content/heart-disease-diagnosis/data/raw/uci_multicenter'),
    Path('/content/data/raw/uci_multicenter'),
    Path('data/raw/uci_multicenter'),
    Path('../data/raw/uci_multicenter'),
]
LOCAL_DATA_DIR = next((path for path in LOCAL_DATA_DIR_CANDIDATES if path.exists()), None)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

def read_uci(site, filename):
    source = (LOCAL_DATA_DIR / filename) if LOCAL_DATA_DIR else f'{BASE_URL}/{filename}'
    frame = pd.read_csv(source, names=COLUMNS, na_values=['?'],
                        skipinitialspace=True).apply(pd.to_numeric, errors='coerce')
    frame[TARGET] = (frame['num'] > 0).astype('int8')
    frame['site'] = site
    return frame[FEATURES + [TARGET, 'site']]

data = pd.concat([read_uci(site, filename) for site, filename in FILES.items()], ignore_index=True)
assert len(data) == 920, f'Expected 920 rows, got {len(data)}'
display(data.groupby('site')[TARGET].agg(['size','sum','mean']).round(4))
source_label = str(LOCAL_DATA_DIR) if LOCAL_DATA_DIR else 'UCI URL fallback'
print('TensorFlow:', tf.__version__, '| Optuna trials per outer fold:', N_TRIALS)
print('Data source:', source_label, '| Evaluation seeds:', EVAL_SEEDS)

In [ ]:
FIXED_PARAMS = {
    'hidden1': 32, 'hidden2': 16, 'dropout': 0.20,
    'learning_rate': 0.001, 'l2_reg': 1e-4, 'batch_size': 32,
    'positive_weight': 1.0,
    'max_epochs': MAX_EPOCHS, 'patience': PATIENCE,
}

def apply_p1(frame):
    out = frame.copy()
    for column in FEATURES:
        out[column] = pd.to_numeric(out[column], errors='coerce')
    for column in ['trestbps', 'chol']:
        out.loc[out[column] <= 0, column] = np.nan
    return out

def make_preprocessor():
    numeric = Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                       ('scaler', StandardScaler())])
    categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
                            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    return ColumnTransformer([('numeric', numeric, NUMERICAL_FEATURES),
                             ('categorical', categorical, CATEGORICAL_FEATURES)])

def make_preprocessor_compat():
    try:
        return make_preprocessor()
    except TypeError:
        numeric = Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                           ('scaler', StandardScaler())])
        categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
                                ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=False))])
        return ColumnTransformer([('numeric', numeric, NUMERICAL_FEATURES),
                                 ('categorical', categorical, CATEGORICAL_FEATURES)])

def prepare(frame):
    ready = apply_p1(frame)
    return ready[FEATURES], ready[TARGET].to_numpy()

def transform_train_test(train_frame, test_frame):
    X_train, y_train = prepare(train_frame)
    X_test, y_test = prepare(test_frame)
    preprocessor = make_preprocessor_compat()
    X_train = preprocessor.fit_transform(X_train).astype('float32')
    X_test = preprocessor.transform(X_test).astype('float32')
    return X_train, y_train, X_test, y_test, preprocessor

def build_mlp(input_dim, params, seed):
    seed_everything(seed)
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(int(params['hidden1']), activation='relu', kernel_regularizer=l2(float(params['l2_reg']))),
        Dropout(float(params['dropout'])),
        Dense(int(params['hidden2']), activation='relu', kernel_regularizer=l2(float(params['l2_reg']))),
        Dropout(float(params['dropout'])),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer=Adam(learning_rate=float(params['learning_rate'])),
                  loss='binary_crossentropy',
                  metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

def fit_mlp(X_train, y_train, X_valid, y_valid, params, seed):
    tf.keras.backend.clear_session()
    model = build_mlp(X_train.shape[1], params, seed)
    reduce_lr = ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5,
                                  patience=max(3, int(params['patience']) // 3),
                                  min_lr=1e-6, verbose=0)
    callback = EarlyStopping(monitor='val_auc', mode='max', patience=int(params['patience']),
                            min_delta=1e-4, restore_best_weights=True, verbose=0)
    class_weight = {0: 1.0, 1: float(params.get('positive_weight', 1.0))}
    history = model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
                        epochs=int(params['max_epochs']), batch_size=int(params['batch_size']),
                        callbacks=[reduce_lr, callback], class_weight=class_weight, verbose=0)
    val_auc_history = history.history.get('val_auc', [])
    best_epoch = int(np.argmax(val_auc_history) + 1) if val_auc_history else len(history.history['loss'])
    return model, history, best_epoch

def select_epochs_on_training(train_frame, params, seed):
    y = train_frame[TARGET].to_numpy()
    groups = train_frame['site'].to_numpy()
    epoch_values = []
    for fold_number, (fit_idx, valid_idx) in enumerate(
        GroupKFold(n_splits=3).split(train_frame, y, groups)):
        fit_frame = train_frame.iloc[fit_idx]
        valid_frame = train_frame.iloc[valid_idx]
        X_fit, y_fit, X_valid, y_valid, _ = transform_train_test(fit_frame, valid_frame)
        _, _, best_epoch = fit_mlp(X_fit, y_fit, X_valid, y_valid, params,
                                seed=seed + fold_number)
        epoch_values.append(best_epoch)
    return max(1, int(np.median(epoch_values))), epoch_values

def fit_mlp_full_training(X_train, y_train, params, epochs, seed):
    tf.keras.backend.clear_session()
    model = build_mlp(X_train.shape[1], params, seed)
    class_weight = {0: 1.0, 1: float(params.get('positive_weight', 1.0))}
    model.fit(X_train, y_train, epochs=int(epochs), batch_size=int(params['batch_size']),
              class_weight=class_weight, verbose=0, shuffle=True)
    return model

def predict_probability(model, X):
    return model.predict(X, batch_size=256, verbose=0).ravel()

def score_probability(y_true, probability):
    prediction = (probability >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {'accuracy': accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'pr_auc': average_precision_score(y_true, probability),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, prediction, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probability) if len(np.unique(y_true)) > 1 else np.nan,
        'brier': brier_score_loss(y_true, probability),
        'false_negatives': int(fn), 'false_positives': int(fp)}

In [ ]:
def suggest_mlp_params(trial):
    return {
        'hidden1': trial.suggest_categorical('hidden1', [16, 32, 64]),
        'hidden2': trial.suggest_categorical('hidden2', [8, 16, 32]),
        'dropout': trial.suggest_float('dropout', 0.0, 0.40),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True),
        'l2_reg': trial.suggest_float('l2_reg', 1e-6, 1e-2, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [16, 32, 64]),
        'positive_weight': trial.suggest_categorical('positive_weight', [0.75, 1.0, 1.25, 1.5, 2.0]),
        'max_epochs': MAX_EPOCHS, 'patience': PATIENCE,
    }

def tune_mlp_on_training(train_frame, seed):
    y = train_frame[TARGET].to_numpy()
    groups = train_frame['site'].to_numpy()
    splits = list(GroupKFold(n_splits=3).split(train_frame, y, groups))

    def objective(trial):
        params = suggest_mlp_params(trial)
        fold_scores = []
        for fold_number, (inner_train_idx, inner_valid_idx) in enumerate(splits):
            inner_train = train_frame.iloc[inner_train_idx]
            inner_valid = train_frame.iloc[inner_valid_idx]
            X_fit, y_fit, X_valid, y_valid, _ = transform_train_test(inner_train, inner_valid)
            model, _, _ = fit_mlp(X_fit, y_fit, X_valid, y_valid, params,
                                 seed=seed + fold_number)
            probability = predict_probability(model, X_valid)
            fold_score = roc_auc_score(y_valid, probability)
            fold_scores.append(fold_score)
            trial.report(float(np.mean(fold_scores)), step=fold_number)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(fold_scores))

    study = optuna.create_study(direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3))
    started = time.perf_counter()
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, show_progress_bar=False)
    return study, time.perf_counter() - started

In [ ]:
outer_records = []
best_param_records = []
trial_records = []
for test_site in FILES:
    # The held-out hospital is not used by tuning, epoch selection, or threshold selection.
    train_frame = data[data['site'] != test_site].reset_index(drop=True)
    test_frame = data[data['site'] == test_site].reset_index(drop=True)

    X_train, y_train, X_test, y_test, _ = transform_train_test(train_frame, test_frame)
    fixed_epoch, fixed_epoch_values = select_epochs_on_training(
        train_frame, FIXED_PARAMS, RANDOM_STATE + 1)

    study, tuning_seconds = tune_mlp_on_training(train_frame, RANDOM_STATE)
    best_params = {**FIXED_PARAMS, **study.best_trial.params}
    best_param_records.append({'test_site': test_site,
        'best_inner_roc_auc': study.best_value,
        'best_params': json.dumps(best_params, sort_keys=True),
        'tuning_seconds': tuning_seconds, 'n_trials': len(study.trials)})
    for trial in study.trials:
        trial_records.append({'test_site': test_site, 'trial_number': trial.number,
            'state': str(trial.state), 'value': trial.value,
            'params': json.dumps(trial.params, sort_keys=True)})

    tuned_epoch, tuned_epoch_values = select_epochs_on_training(
        train_frame, best_params, RANDOM_STATE + 2)
    for eval_seed in EVAL_SEEDS:
        fixed_started = time.perf_counter()
        fixed_model = fit_mlp_full_training(
            X_train, y_train, FIXED_PARAMS, fixed_epoch, eval_seed)
        fixed_fit_seconds = time.perf_counter() - fixed_started
        fixed_probability = predict_probability(fixed_model, X_test)
        outer_records.append({'validation': 'LOCO', 'test_site': test_site,
            'seed': eval_seed, 'configuration': 'F0_P1_fixed', 'model': 'MLP',
            'tuning_seconds': 0.0, 'fit_seconds': fixed_fit_seconds,
            'best_inner_roc_auc': np.nan, 'best_epoch': fixed_epoch,
            'epoch_values': json.dumps(fixed_epoch_values),
            'best_params': json.dumps(FIXED_PARAMS, sort_keys=True),
            **score_probability(y_test, fixed_probability)})

        tuned_started = time.perf_counter()
        tuned_model = fit_mlp_full_training(
            X_train, y_train, best_params, tuned_epoch, eval_seed)
        tuned_fit_seconds = time.perf_counter() - tuned_started
        tuned_probability = predict_probability(tuned_model, X_test)
        outer_records.append({'validation': 'LOCO', 'test_site': test_site,
            'seed': eval_seed, 'configuration': 'F0_P1_optuna_nested', 'model': 'MLP',
            'tuning_seconds': tuning_seconds, 'fit_seconds': tuned_fit_seconds,
            'best_inner_roc_auc': study.best_value, 'best_epoch': tuned_epoch,
            'epoch_values': json.dumps(tuned_epoch_values),
            'best_params': json.dumps(best_params, sort_keys=True),
            **score_probability(y_test, tuned_probability)})
    print('Completed outer test site:', test_site)

results_df = pd.DataFrame(outer_records)
best_params_df = pd.DataFrame(best_param_records)
trials_df = pd.DataFrame(trial_records)
display(results_df.round(4))
display(best_params_df)

In [ ]:
summary = results_df.groupby(['configuration', 'model']).agg(
    eval_rows=('seed', 'size'), folds=('test_site', 'nunique'),
    seeds=('seed', 'nunique'), roc_auc_mean=('roc_auc', 'mean'),
    roc_auc_std=('roc_auc', 'std'), pr_auc_mean=('pr_auc', 'mean'),
    recall_mean=('recall', 'mean'), recall_std=('recall', 'std'),
    specificity_mean=('specificity', 'mean'), f1_mean=('f1', 'mean'),
    brier_mean=('brier', 'mean'),
    false_negatives_mean_per_fold=('false_negatives', 'mean'),
    false_negatives_total=('false_negatives', 'sum'),
    false_positives_mean_per_fold=('false_positives', 'mean'),
    false_positives_total=('false_positives', 'sum'),
    best_inner_roc_auc_mean=('best_inner_roc_auc', 'mean'),
    best_epoch_mean=('best_epoch', 'mean'),
    tuning_seconds_mean=('tuning_seconds', 'mean'),
    fit_seconds_mean=('fit_seconds', 'mean')).reset_index()

# Worst-site metrics average over seeds first, then take the weakest hospital.
site_summary = results_df.groupby(['configuration', 'model', 'test_site']).agg(
    roc_auc=('roc_auc', 'mean'), pr_auc=('pr_auc', 'mean'),
    recall=('recall', 'mean'), brier=('brier', 'mean')).reset_index()
worst_site = site_summary.groupby(['configuration', 'model']).agg(
    roc_auc_worst=('roc_auc', 'min'), pr_auc_worst=('pr_auc', 'min'),
    recall_worst=('recall', 'min'), brier_worst=('brier', 'max')).reset_index()

seed_summary = results_df.groupby(['configuration', 'model', 'seed']).agg(
    roc_auc_mean=('roc_auc', 'mean'), pr_auc_mean=('pr_auc', 'mean'),
    recall_mean=('recall', 'mean'), brier_mean=('brier', 'mean')).reset_index()
seed_variance = seed_summary.groupby(['configuration', 'model']).agg(
    roc_auc_seed_std=('roc_auc_mean', 'std'),
    pr_auc_seed_std=('pr_auc_mean', 'std'),
    recall_seed_std=('recall_mean', 'std'),
    brier_seed_std=('brier_mean', 'std')).reset_index()
summary = summary.merge(worst_site, on=['configuration', 'model'])
summary = summary.merge(seed_variance, on=['configuration', 'model'])

baseline = summary[summary['configuration'] == 'F0_P1_fixed'].set_index('model')
delta = summary.copy()
for metric in ['roc_auc_mean', 'roc_auc_worst', 'pr_auc_mean', 'recall_mean',
                'recall_worst', 'specificity_mean', 'brier_mean',
                'false_negatives_total', 'false_positives_total']:
    delta[f'delta_vs_fixed_{metric}'] = delta.apply(
        lambda row: row[metric] - baseline.loc[row['model'], metric], axis=1)

display(summary.round(6))
display(delta.round(6))
display(site_summary.sort_values(['configuration', 'roc_auc']))
display(seed_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=summary, x='model', y='roc_auc_mean', hue='configuration', ax=axes[0])
axes[0].set_title('MLP nested LOCO mean ROC-AUC')
sns.barplot(data=summary, x='model', y='recall_mean', hue='configuration', ax=axes[1])
axes[1].set_title('MLP nested LOCO mean recall at threshold 0.50')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'mlp_loco_summary.png', dpi=180, bbox_inches='tight')
plt.show()

## So sánh với Lab 22

Các dòng Logistic Regression và LightGBM dưới đây là kết quả LOCO đã báo cáo từ Lab 22. Kết quả MLP được lấy trực tiếp từ `summary` của notebook này.

In [ ]:
LAB22_REFERENCE = pd.DataFrame([
    {'model': 'LightGBM', 'configuration': 'F0_P1_fixed', 'roc_auc_mean': 0.781092,
     'roc_auc_worst': 0.684564, 'recall_mean': 0.786385, 'recall_worst': 0.711409,
     'specificity_mean': 0.662842, 'f1_mean': 0.775555, 'brier_mean': 0.187057,
     'false_negatives_mean_per_fold': 27.50},
    {'model': 'LightGBM', 'configuration': 'F0_P1_optuna_nested', 'roc_auc_mean': 0.799137,
     'roc_auc_worst': 0.716147, 'recall_mean': 0.809457, 'recall_worst': 0.711409,
     'specificity_mean': 0.584024, 'f1_mean': 0.758738, 'brier_mean': 0.183609,
     'false_negatives_mean_per_fold': 24.50},
    {'model': 'Logistic Regression', 'configuration': 'F0_P1_fixed', 'roc_auc_mean': 0.803802,
     'roc_auc_worst': 0.712725, 'recall_mean': 0.781991, 'recall_worst': 0.676259,
     'specificity_mean': 0.682130, 'f1_mean': 0.807953, 'brier_mean': 0.153930,
     'false_negatives_mean_per_fold': 28.25},
    {'model': 'Logistic Regression', 'configuration': 'F0_P1_optuna_nested', 'roc_auc_mean': 0.808415,
     'roc_auc_worst': 0.729964, 'recall_mean': 0.815676, 'recall_worst': 0.769784,
     'specificity_mean': 0.615126, 'f1_mean': 0.791102, 'brier_mean': 0.165409,
     'false_negatives_mean_per_fold': 23.75},
])

common_columns = ['model', 'configuration', 'roc_auc_mean', 'roc_auc_worst',
    'recall_mean', 'recall_worst', 'specificity_mean', 'f1_mean', 'brier_mean',
    'false_negatives_mean_per_fold']
mlp_reference = summary[common_columns].copy()
comparison = pd.concat([LAB22_REFERENCE[common_columns], mlp_reference], ignore_index=True)
display(comparison.sort_values(['roc_auc_mean'], ascending=False).round(6))
comparison.to_csv(OUTPUT_DIR / 'mlp_vs_lab22_loco_comparison.csv', index=False)

## Diễn giải

- Nếu MLP không vượt Logistic Regression/LightGBM trên worst-cohort AUC và recall, giữ mô hình truyền thống làm ứng viên chính.
- Nếu MLP có AUC cao nhưng Brier hoặc specificity kém, cần calibration và threshold tuning trước khi đánh giá ứng dụng.
- Do dữ liệu chỉ có 920 dòng, một kết quả MLP tốt hơn nhỏ không đủ để kết luận Deep Learning tốt hơn; cần xem cả độ ổn định giữa bốn cohort.
- Không dùng kết quả MLP này để tuyên bố xác nhận lâm sàng; đây vẫn là thực nghiệm robustness trên UCI.

In [ ]:
results_df.to_csv(OUTPUT_DIR / 'mlp_loco_results.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'mlp_loco_summary.csv', index=False)
delta.to_csv(OUTPUT_DIR / 'mlp_loco_delta_vs_fixed.csv', index=False)
best_params_df.to_csv(OUTPUT_DIR / 'mlp_best_params_by_outer_fold.csv', index=False)
trials_df.to_csv(OUTPUT_DIR / 'mlp_optuna_trial_history.csv', index=False)
site_summary.to_csv(OUTPUT_DIR / 'mlp_loco_site_summary.csv', index=False)
seed_summary.to_csv(OUTPUT_DIR / 'mlp_loco_seed_summary.csv', index=False)
seed_variance.to_csv(OUTPUT_DIR / 'mlp_loco_seed_variance.csv', index=False)
run_config = {
    'dataset_rows': 920,
    'data_source': source_label,
    'validation': 'outer LOCO + inner GroupKFold by hospital',
    'preprocessing': 'P1_sentinel_aware with dense OneHotEncoder for Keras',
    'model': 'small Keras MLP',
    'fixed_architecture': 'input -> 32 -> 16 -> 1',
    'max_epochs': MAX_EPOCHS, 'early_stopping_patience': PATIENCE,
    'n_trials_per_outer_fold': N_TRIALS,
    'optimization_metric': 'inner mean ROC-AUC',
    'tuning_seed': RANDOM_STATE,
    'evaluation_seeds': list(EVAL_SEEDS),
    'threshold': THRESHOLD, 'threshold_tuning': 'deferred',
    'class_weight': 'positive_weight tuned in inner CV; applied to train rows only',
    'synthetic_data': 'not used'
}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
zip_path = shutil.make_archive('/content/uci_multicenter_mlp_loco_results', 'zip', OUTPUT_DIR)
print('Saved:', OUTPUT_DIR, zip_path)